In [1]:
from langchain_community.document_loaders.json_loader import JSONLoader
from pathlib import Path
from pprint import pp

C:\Users\kamra\AppData\Local\Temp\ipykernel_13160\3260479200.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.json_loader import JSONLoader


In [2]:
# create the path for json file

file_path = Path("../knowledge-source/apparels.json")

file_path.exists()

True

In [3]:
# from the loader

loader = JSONLoader(file_path = file_path.as_posix(),
                    jq_schema=".products[].Description") # fetching description for each page

In [4]:
# documents

documents = loader.load()

In [5]:
print(documents)

[Document(metadata={'source': 'E:\\RAG\\RAG-DOCUMENT-LOADERS\\knowledge-source\\apparels.json', 'seq_num': 1}, page_content='Oversize-fit coat made of a viscose blend fabric. Notch lapel collar and long sleeves with buttoned cuffs.'), Document(metadata={'source': 'E:\\RAG\\RAG-DOCUMENT-LOADERS\\knowledge-source\\apparels.json', 'seq_num': 2}, page_content='Trench coat made of technical fabric with a velvety finish. Notch lapel collar and long sleeves.'), Document(metadata={'source': 'E:\\RAG\\RAG-DOCUMENT-LOADERS\\knowledge-source\\apparels.json', 'seq_num': 3}, page_content='Parka made of technical fabric, padded on inside. High neck with a hood and long sleeves with elasticated cuffs.'), Document(metadata={'source': 'E:\\RAG\\RAG-DOCUMENT-LOADERS\\knowledge-source\\apparels.json', 'seq_num': 4}, page_content='The first Vibrant Leather for le benjamin with its fruity hint of pineapple that gives it modernity, accompanied by cedar notes, adding a woody touch of rejuvenation.'), Documen

In [ ]:
# I will run a for loop to print the page content. 

for doc in documents:
    print(doc.page_content,end="\n\n")

Oversize-fit coat made of a viscose blend fabric. Notch lapel collar and long sleeves with buttoned cuffs.

Trench coat made of technical fabric with a velvety finish. Notch lapel collar and long sleeves.

Parka made of technical fabric, padded on inside. High neck with a hood and long sleeves with elasticated cuffs.

The first Vibrant Leather for le benjamin with its fruity hint of pineapple that gives it modernity, accompanied by cedar notes, adding a woody touch of rejuvenation.

Multicoloured backpack. Soft construction. Featuring one main zip pocket and a medium-sized zip pocket on the inside.



- if i show u thr meta data of documents of 1st object
- So, the metadata currently contains only the source and sequence information.
- What if I want to add other information to the metadata as well?
- For example, product_id, manufacturer, image URL, etc.
- I want to add all this information to the metadata.
- I also want to add the description to the page_content.
- For this, instead of passing a specific field in the jq_schema parameter,
- we pass the product array inside the jq_schema.
- Now, all the JSON content will be extracted because we are not selecting any particular field.

In [ ]:
# Create a function called metadata that takes two inputs and returns one dictionary as output.
def metadata_func(record: dict, metadata:dict) -> dict:
    metadata["product_name"]=record["productName"]
    metadata["category"] = record["category"]
    # I want to add the price to the metadata.
    # The price value will be extracted from the record.
    metadata["price"] = record["price"]

    # You can also delete or modify existing values.
    # For example, let's say I want to delete 'seq_num': 1. 
    del metadata["seq_num"] # The 'seq_num' has been removed. Check cell 26.

    return metadata


In [ ]:
# For example, you need to define what your source and metadata should be.
# Here, we define the source as the content_key, which is "description".
loader1 = JSONLoader(file_path = file_path.as_posix(),
                    jq_schema=".products[]",
                    content_key = "Description",
                    metadata_func = metadata_func)

In [24]:
documents1 = loader1.load()

In [25]:
# load the data
for doc in documents1:
    print(doc.page_content,end="\n\n")

Oversize-fit coat made of a viscose blend fabric. Notch lapel collar and long sleeves with buttoned cuffs.

Trench coat made of technical fabric with a velvety finish. Notch lapel collar and long sleeves.

Parka made of technical fabric, padded on inside. High neck with a hood and long sleeves with elasticated cuffs.

The first Vibrant Leather for le benjamin with its fruity hint of pineapple that gives it modernity, accompanied by cedar notes, adding a woody touch of rejuvenation.

Multicoloured backpack. Soft construction. Featuring one main zip pocket and a medium-sized zip pocket on the inside.



In [26]:
# if i show u the meta data

documents1[0].metadata

# 

{'source': 'E:\\RAG\\RAG-DOCUMENT-LOADERS\\knowledge-source\\apparels.json',
 'product_name': 'PINSTRIPE COAT',
 'category': 'Men Cloths',
 'price': 4900}

### In a nutshell:

- 1. First, we need to pass jq_schema inside the JSONLoader because we are using the jq parser.
     Once we pass the schema, our documents will be extracted.

- 2. Our JSON file contains a key called "product", which is an array.
     Inside this array, we have JSON objects that we want to extract.

- 3. Now, we can decide which information should be stored in page_content and which should be stored   in metadata.

- 4. For the page_content field, we use a for loop to extract and add the description.
- 5. To add information to metadata, we can extract the required fields using the metadata function.
- 6. The metadata function takes two dictionaries as input:
     The first is the original JSON record, and the second is the existing metadata.
     It returns the updated metadata dictionary.